In [ ]:
import os
import time
import json
import joblib
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

IMG_ROOT = '../data/Images'
IMG_SIZE = (128, 128)
label_map = {
    'Negative': 'negative',
    'Neutral':  'neutral',
    'positive': 'positive',
}

## Load and prepare data

In [ ]:
def load_image(path):
    img = Image.open(path).convert('RGB')
    img = img.resize(IMG_SIZE, Image.LANCZOS)
    return np.array(img, dtype='float32') / 255.0

records = []
for folder, label in label_map.items():
    folder_path = os.path.join(IMG_ROOT, folder)
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            records.append({'path': os.path.join(folder_path, fname), 'label': label})

df = pd.DataFrame(records)

# Test set locked in — identical to all other image notebooks
paths_trainval, paths_test, y_trainval, y_test = train_test_split(
    df['path'], df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Val set carved from training portion only
paths_train, paths_val, y_train, y_val = train_test_split(
    paths_trainval, y_trainval,
    test_size=0.2,
    random_state=42,
    stratify=y_trainval
)

print(f'Train: {len(paths_train)}  Val: {len(paths_val)}  Test: {len(paths_test)}')
print('Loading images...')

X_train = np.array([load_image(p) for p in paths_train]).reshape(len(paths_train), -1)
X_val   = np.array([load_image(p) for p in paths_val]).reshape(len(paths_val), -1)
X_test  = np.array([load_image(p) for p in paths_test]).reshape(len(paths_test), -1)
X_trainval = np.array([load_image(p) for p in paths_trainval]).reshape(len(paths_trainval), -1)

print(f'Feature vector size: {X_train.shape[1]:,}')  # 128*128*3 = 49,152

## Tune C on validation set

In [ ]:
t0 = time.time()
C_values = [0.001, 0.01, 0.1, 1, 10]
val_results = []

for C in C_values:
    clf = LogisticRegression(C=C, max_iter=1000, solver='saga')
    clf.fit(X_train, y_train)
    val_acc = accuracy_score(y_val, clf.predict(X_val))
    val_results.append({'C': C, 'val_acc': val_acc})
    print(f'C={C:<6}  Val Accuracy: {val_acc:.3f}')

best_C = max(val_results, key=lambda x: x['val_acc'])['C']
print(f'\nBest C: {best_C}')

## Retrain on full train+val, evaluate on test

In [ ]:
best_clf = LogisticRegression(C=best_C, max_iter=1000, solver='saga')
best_clf.fit(X_trainval, y_trainval)
y_pred = best_clf.predict(X_test)
runtime = time.time() - t0

print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(f'Runtime:  {runtime:.2f}s')
print()
print(classification_report(y_test, y_pred))

report = classification_report(y_test, y_pred, output_dict=True)
meta = {
    'model': 'logistic_regression',
    'accuracy': report['accuracy'],
    'macro_f1': report['macro avg']['f1-score'],
    'negative_f1': report['negative']['f1-score'],
    'neutral_f1': report['neutral']['f1-score'],
    'positive_f1': report['positive']['f1-score'],
    'runtime_seconds': runtime
}
joblib.dump(best_clf, '../models/images/fits/logistic_regression.pkl')
with open('../models/images/json/logistic_regression_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved.')